In [20]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import re

from tqdm import trange, tqdm
from transformers import AutoTokenizer

GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')
print(device)


cuda:0


In [21]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

In [22]:
train = pd.read_csv('./sentiment_dataset/train.csv')
train = train.drop('selected_text', axis=1)
train = train.drop('Time of Tweet', axis=1)
train = train.drop('Age of User', axis=1)
train = train.drop('Country', axis=1)
train = train.drop('Population -2020', axis=1)
train = train.drop('Land Area (Km)', axis=1)
train = train.drop('Density (P/Km)', axis=1)
train = train.drop('textID', axis=1)
train['sentiment'] = train['sentiment'].map(lambda x: {'positive': 2, 'neutral' : 1, 'negative' : 0}[x])
train

,text,sentiment
0,"I`d have responded, if I were going",1
1,Sooo SAD I will miss you here in San Diego!!!,0
2,my boss is bullying me...,0
3,what interview! leave me alone,0
4,"Sons of ****, why couldn`t they put them on t...",0
...,...,...
27476,wish we could come see u on Denver husband l...,0
27477,I`ve wondered about rake to. The client has ...,0
27478,Yay good for both of you. Enjoy the break - y...,2
27479,But it was worth it ****.,2


In [23]:
test = pd.read_csv('./sentiment_dataset/test.csv')
test = test.drop('Time of Tweet', axis=1)
test = test.drop('Age of User', axis=1)
test = test.drop('Country', axis=1)
test = test.drop('Population -2020', axis=1)
test = test.drop('Land Area (Km)', axis=1)
test = test.drop('Density (P/Km)', axis=1)
test = test.drop('textID', axis=1)
test['sentiment'] = test['sentiment'].map(lambda x: {'positive': 2, 'neutral' : 1, 'negative' : 0}[x])
test

,text,sentiment
0,Last session of the day http://twitpic.com/67ezh,1
1,Shanghai is also really exciting (precisely -...,2
2,"Recession hit Veronique Branquinho, she has to...",0
3,happy bday!,2
4,http://twitpic.com/4w75p - I like it!!,2
...,...,...
3529,"its at 3 am, im very tired but i can`t sleep ...",0
3530,All alone in this old house again. Thanks for...,2
3531,I know what you mean. My little dog is sinkin...,0
3532,_sutra what is your next youtube video gonna b...,2


In [24]:
train = train.dropna(subset=['text'])
test = test.dropna(subset=['text'])


In [25]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [26]:
tokenized_train = tokenizer(list(train['text']), padding=True, truncation=True, return_tensors='np')
tokenized_test = tokenizer(list(test['text']), padding=True, truncation=True, return_tensors='np')

In [27]:
train_data = tokenized_train['input_ids']
test_data = tokenized_test['input_ids']

In [28]:
class tweet_dataset(Dataset):
    def __init__ (self, tweet, sentiment) :
        self.tweet = tweet
        self.sentiment = sentiment

    def __len__ (self):
        return len(self.tweet)
    
    def __getitem__(self, index):
        return self.tweet[index], self.sentiment.iloc[index]

In [29]:
train_dataset = DataLoader(tweet_dataset(train_data, train['sentiment']), batch_size=32, shuffle=True)
test_dataset = DataLoader(tweet_dataset(test_data, test['sentiment']), batch_size=32, shuffle=False)

In [30]:
class rnn(nn.Module) :
    def __init__ (self, vocab_size, input_size, hidden_size, output_size, pad_idx):
        super(rnn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size, pad_idx)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(2*hidden_size, output_size)

    def forward (self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        final_memory, _ = torch.max(out, dim=1)

        return self.fc(final_memory)

In [31]:
model = rnn(len(tokenizer), 128, 128, 3, tokenizer.pad_token_id).to(device)
criterion = nn.CrossEntropyLoss() # ignore_index=tokenizer.pad_token_id
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 20 
print(model)

rnn(
  (embedding): Embedding(30522, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)


In [32]:
def train_epoch(model, train_loader, criterion, optimizer, loss_logger):
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        
        outputs = model(data.to(device))
        loss = criterion(outputs, target.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_logger.append(loss.item())

    return model, optimizer, loss_logger

In [33]:
def test_model(model, test_loader, criterion, loss_logger):
    with torch.no_grad():
        correct_predictions = 0
        total_predictions = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(test_loader, desc="Testing", leave=False)):   
            
            outputs = model(data.to(device))        
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == target.to(device)).sum().item()
            total_predictions += target.shape[0]
            loss = criterion(outputs, target.to(device))
            loss_logger.append(loss.item())
            
        acc = (correct_predictions/total_predictions) * 100.0
        return loss_logger, acc

In [34]:
train_loss = []
test_loss  = []
test_acc   = []

In [35]:
for i in trange(num_epochs, desc="Epoch", leave=False):
    model, optimizer, train_loss = train_epoch(model, train_dataset, criterion, optimizer, train_loss)
    
    test_loss, acc = test_model(model, test_dataset, criterion, test_loss)
    test_acc.append(acc)

print("Final Accuracy: %.2f%%" % acc)

Final Accuracy: 70.12%
